# Cleaning patients

In [104]:
#pip install -U ipython-sql

In [105]:
# DO NOT EDIT THIS CELL
import sqlite3
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [106]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [107]:

%sql sqlite:///carepulse.db

In [108]:
%%sql
SELECT * FROM patients LIMIT 5;

 * sqlite:///carepulse.db
Done.


patient_id,first_name,last_name,age,gender,province,city,employment_status,annual_income,registration_date
1,Anele,Van Wyk,60.0,Male,Limpopo,Limpopo City C,Unemployed,16156.03,2025-03-12
2,Ayanda,Nkosi,23.0,Female,North west,North West City C,Student,38700.75,2026-05-18
3,Ayanda,Nkosi,69.0,Male,North west,North West City B,Unemployed,48448.73,2025-04-18
4,Liam,Naidoo,32.0,Female,Western cape,Western Cape City A,Unemployed,18668.75,2025-01-26
5,Thabo,Botha,61.0,Female,Kwazulu-natal,KwaZulu-Natal City B,Unemployed,18938.98,2025-03-09


In [109]:
%%sql
PRAGMA table_info(Patients);

 * sqlite:///carepulse.db
Done.


cid,name,type,notnull,dflt_value,pk
0,patient_id,INTEGER,0,None,0
1,first_name,TEXT,0,None,0
2,last_name,TEXT,0,None,0
3,age,REAL,0,None,0
4,gender,TEXT,0,None,0
5,province,TEXT,0,None,0
6,city,TEXT,0,None,0
7,employment_status,TEXT,0,None,0
8,annual_income,TEXT,0,None,0
9,registration_date,TEXT,0,None,0


In [133]:
%%sql
# we rebuilding the table becuase we wanted to change data types for age and income
-- Create new table
CREATE TABLE patients_new (
    patient_id INTEGER,
    first_name TEXT,
    last_name TEXT,
    age INTEGER,
    gender TEXT,
    province TEXT,
    city TEXT,
    employment_status TEXT,
    annual_income REAL,
    registration_date TEXT
);

-- Copy data with type conversions
INSERT INTO patients_new (
    patient_id,
    first_name,
    last_name,
    age,
    gender,
    province,
    city,
    employment_status,
    annual_income,
    registration_date
)
SELECT
    patient_id,
    first_name,
    last_name,
    CAST(age AS INTEGER),
    gender,
    province,
    city,
    employment_status,
    CAST(annual_income AS REAL),
    registration_date
FROM patients;

 * sqlite:///carepulse.db
(sqlite3.OperationalError) unrecognized token: "#"
[SQL: # we rebuilding the table becuase we wanted to change data types for age and income
-- Create new table
CREATE TABLE patients_new (
    patient_id INTEGER,
    first_name TEXT,
    last_name TEXT,
    age INTEGER,
    gender TEXT,
    province TEXT,
    city TEXT,
    employment_status TEXT,
    annual_income REAL,
    registration_date TEXT
);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [126]:
%%sql
PRAGMA table_info(patients_new);

 * sqlite:///carepulse.db
Done.


cid,name,type,notnull,dflt_value,pk
0,patient_id,INTEGER,0,None,0
1,first_name,TEXT,0,None,0
2,last_name,TEXT,0,None,0
3,age,INTEGER,0,None,0
4,gender,TEXT,0,None,0
5,province,TEXT,0,None,0
6,city,TEXT,0,None,0
7,employment_status,TEXT,0,None,0
8,annual_income,REAL,0,None,0
9,registration_date,TEXT,0,None,0


In [129]:
%%sql
SELECT * FROM patients_new WHERE age IS NULL LIMIT 5;


 * sqlite:///carepulse.db
Done.


patient_id,first_name,last_name,age,gender,province,city,employment_status,annual_income,registration_date
27,Sipho,Dlamini,None,Female,Mpumalanga,Mpumalanga City C,SELF-EMPLOYED,28561.73,2025-05-18
225,Sipho,Mokoena,None,Other,Northern cape,Northern Cape City B,Retired,8585.96,2026-01-03
281,Noah,Mokoena,None,Female,Eastern cape,Eastern Cape City A,Retired,22870.89,2026-05-10
415,Sipho,Jacobs,None,Female,Northern cape,Northern Cape City C,Self-employed,20412.57,2026-02-07
422,Nomsa,Naidoo,None,Male,Western cape,Western Cape City A,STUDENT,4769.21,2026-02-10


In [137]:
%%sql
SELECT
    gender,
    CAST(ROUND(AVG(age), 0) AS INTEGER) AS average_age
FROM patients_new
GROUP BY gender;

 * sqlite:///carepulse.db
Done.


gender,average_age
Female,54
Male,53
Other,53


In [134]:
# %%sql
# UPDATE patients
# SET age = CAST(age AS INTEGER);


In [114]:
%%sql
SELECT  distinct(gender) FROM patients ;

 * sqlite:///carepulse.db
Done.


gender
Male
Female
Other


In [115]:
%%sql
SELECT COUNT(*)
FROM patients
WHERE gender  IS NULL ;

 * sqlite:///carepulse.db
Done.


COUNT(*)
0


In [116]:
%%sql
UPDATE patients
SET gender = 'Other'
WHERE gender IS NULL;


 * sqlite:///carepulse.db
0 rows affected.


[]

In [117]:
%%sql
UPDATE patients
SET gender =
    UPPER(SUBSTR(TRIM(gender), 1, 1)) ||
    LOWER(SUBSTR(TRIM(gender), 2));

 * sqlite:///carepulse.db
100700 rows affected.


[]

In [118]:
%%sql
SELECT  distinct(province) FROM patients LIMIT 10;

 * sqlite:///carepulse.db
Done.


province
Limpopo
North west
Western cape
Kwazulu-natal
Gauteng
Northern cape
Free State
Mpumalanga
Eastern cape
Free state


In [119]:
%%sql
SELECT *
FROM patients
WHERE province  IS NULL LIMIT 10;

 * sqlite:///carepulse.db
Done.


patient_id,first_name,last_name,age,gender,province,city,employment_status,annual_income,registration_date


In [120]:
%%sql
UPDATE patients
SET province =
    CASE
        WHEN city LIKE 'Free State%' THEN 'Free State'
        WHEN city LIKE 'Mpumalanga%' THEN 'Mpumalanga'
        WHEN city LIKE 'Gauteng%' THEN 'Gauteng'
        WHEN city LIKE 'North West%' THEN 'North West'
        WHEN city LIKE 'Northern Cape%' THEN 'Northern Cape'
        WHEN city LIKE 'Western Cape%' THEN 'Western Cape'
        WHEN city LIKE 'Eastern Cape%' THEN 'Eastern Cape'
        WHEN city LIKE 'KwaZulu-Natal%' THEN 'KwaZulu-Natal'
        WHEN city LIKE 'Limpopo%' THEN 'Limpopo'
        ELSE province
    END
WHERE province IS NULL;

 * sqlite:///carepulse.db
0 rows affected.


[]